# Clase 5. Predicción sin modelo: Monte Carlo y Diferencias Temporales

**Duración estimada:** 90 minutos
**Capítulo del manual:** [06_prediccion_sin_modelo.md](../manual/06_prediccion_sin_modelo.md)

## De ecuaciones conocidas a pura experiencia

En la clase anterior resolviste el Proceso de Decisión de Markov (PDM) con Programación Dinámica (DP): un conjunto de métodos potentes, pero que exige conocer de antemano el modelo completo del entorno, es decir, las probabilidades de transición y la función de recompensa. En un problema real —el riego de un cultivo, el mantenimiento de una flota pesquera, un protocolo de cuidados— casi nunca dispones de ese modelo exacto.

Hoy cambias de enfoque: aprendes **sin modelo** (model-free). En vez de calcular el valor de cada estado con ecuaciones conocidas, lo estimas a partir de la experiencia, episodio tras episodio, como aprendería cualquier persona un oficio nuevo por observación. Vas a programar las dos familias de métodos que hacen eso: Monte Carlo (MC) y las Diferencias Temporales (TD, del inglés Temporal-Difference). Ambas resuelven la misma tarea —la **predicción**: estimar Vπ(s) sin conocer el modelo— pero llegan ahí por caminos distintos, con consecuencias prácticas que vas a comparar en código dentro de un momento.

### Objetivos de hoy

- Explicar qué significa "predicción sin modelo" y en qué se diferencia de la Programación Dinámica
- Programar Monte Carlo de primera visita para estimar Vπ a partir de episodios completos
- Programar TD(0) y entender el bootstrapping: actualizar una estimación con otra estimación
- Comparar la velocidad de convergencia de ambos métodos y relacionarla con el balance entre sesgo y varianza

> 🎙️ **Guion de clase:** abre preguntando "si nadie os diera el mapa de tráfico, ¿cómo estimaríais cuánto tarda un trayecto?". Deja que respondan "conduciéndolo varias veces" antes de nombrar Monte Carlo: es la intuición que vas a formalizar hoy.


## 1. Qué significa "predicción" sin modelo

Recuerda que Vπ(s) es el retorno esperado si sigues la política π desde el estado s. Con DP calculabas ese valor resolviendo ecuaciones sobre el modelo. Sin modelo, la única fuente de información que te queda es la experiencia real: dejas que el agente actúe según π, observas las recompensas que recibe y, con esos datos, estimas Vπ(s).

Es la diferencia entre calcular la duración media de un trayecto con un mapa de tráfico perfecto y estimarla cronometrando el trayecto muchas veces.


## 2. Monte Carlo: aprende del episodio completo

MC es el más directo de los dos métodos: deja que el episodio termine, calcula el retorno total obtenido desde cada estado visitado y usa ese retorno como muestra del valor real. Repite esto con muchos episodios y promedia. Por la ley de los grandes números, ese promedio converge al verdadero Vπ(s).

Piensa en el trayecto de casa al trabajo: el enfoque MC espera a que llegues, anota el tiempo total y actualiza tu estimación con ese dato completo. Cuantos más días repitas el trayecto, mejor será tu estimación, pero necesitas llegar cada vez para aprender algo: si el episodio no termina, MC no tiene nada que promediar.

Dentro de MC existen dos variantes, según cuántas veces cuentes cada estado dentro de un mismo episodio:

- **Primera visita** (first-visit): usas solo el retorno desde la primera vez que aparece un estado en el episodio, aunque se repita más adelante
- **Cada visita** (every-visit): usas el retorno desde cada ocurrencia del estado, no solo la primera

Ambas variantes convergen al mismo valor con suficientes episodios; hoy programas la primera.

> 🎙️ **Guion de clase:** pregunta "¿qué pasaría si un estado aparece tres veces en el mismo episodio?" antes de explicar la diferencia entre primera visita y cada visita. Deja que intuyan que "primera visita" es la opción más simple de programar.


## 3. Diferencias Temporales: adivina a partir de tu propia adivinanza

TD(0) resuelve el mismo problema con una idea distinta: no esperes al final del episodio, actualiza tu estimación después de cada paso. Para eso combina la recompensa que acabas de observar con tu propia estimación actual del valor del siguiente estado, en lugar de esperar el retorno real completo. Esta técnica se llama **bootstrapping**: usas una estimación para corregir otra estimación.

Siguiendo la analogía del trayecto: en vez de esperar a llegar para juzgar si el viaje va bien, ajustas tu previsión en cada tramo. Al salir bajo la lluvia, revisas al alza tu estimación. Al incorporarte a una autopista sin atascos, la revisas a la baja. No necesitas llegar a casa para aprender algo útil en cada tramo.

La regla de actualización de TD(0) es esta:

```
V(S) ← V(S) + α · [R + γ·V(S') − V(S)]
```

Donde α es la tasa de aprendizaje, R es la recompensa observada al dar el paso, γ es el factor de descuento y V(S') es tu estimación actual del siguiente estado (cero si S' es terminal). El término entre corchetes es el **error TD**: la diferencia entre lo que acabas de observar más lo que crees que vale el resto del camino, y lo que creías antes de dar el paso.

Como TD no espera al final, funciona incluso con episodios incompletos o tareas que no terminan nunca, algo que MC no puede hacer.

> 🎙️ **Guion de clase:** escribe la fórmula en la pizarra y señala cada símbolo mientras la lees en voz alta: "el nuevo valor es el viejo valor, más un poquito del error entre lo que acabo de ver y lo que ya creía". Es la frase que la clase repite dentro de un momento, ya en código.


## 4. Sesgo y varianza: la gran disyuntiva

Esta diferencia de mecánica tiene una consecuencia estadística directa:

- MC promedia retornos completos y reales, así que su estimación es **no sesgada**: en promedio acierta el valor verdadero. El precio es una **varianza alta**, porque cada retorno depende de toda la secuencia de acciones y recompensas aleatorias hasta el final del episodio
- TD actualiza con una sola recompensa observada más una estimación todavía imperfecta del resto, así que es **sesgado** mientras esa estimación no haya convergido. A cambio tiene **varianza baja**, porque cada actualización depende solo de un paso

Con infinitos episodios y una tasa de aprendizaje que decae adecuadamente, ambos métodos convergen al mismo Vπ(s). Con datos finitos —la situación real casi siempre— pueden dar respuestas distintas, y suele compensar aceptar algo de sesgo a cambio de mucha menos varianza. Por eso se usa tanto TD en la práctica: aprende rápido y con estimaciones estables, aunque tarde un poco en "quitarse" el sesgo inicial.


## 5. Manos a la obra: el paseo aleatorio

Vas a comparar ambos métodos en el **paseo aleatorio** (random walk), un entorno mínimo pero muy ilustrativo: una cadena de cinco estados no terminales, con un estado terminal a cada extremo. Desde el estado central, en cada paso te mueves a la izquierda o a la derecha con la misma probabilidad. Si llegas al extremo derecho recibes una recompensa de uno; si llegas al extremo izquierdo, recibes cero.

Es un entorno tan simple que puedes calcular el valor real de cada estado a mano, lo que te permite medir con precisión qué tan rápido converge cada método. Construye el código célula a célula, ejecutando cada una antes de pasar a la siguiente.

> 🎙️ **Guion de clase:** dibuja en la pizarra la cadena de siete estados (dos terminales, cinco internos) y marca el estado inicial en el centro. Es la imagen que la clase tiene presente durante el resto de la sesión.


In [ ]:
# Solo necesitas numpy y matplotlib: ambos ya vienen preinstalados en Google Colab
import numpy as np
import matplotlib.pyplot as plt


### El entorno: definición

Numera los estados de cero a seis: 0 y 6 son terminales (izquierda y derecha); del uno al cinco son los estados internos A, B, C, D, E, y el episodio siempre arranca en el estado central, C.

Como el entorno es tan simple, el valor real de cada estado interno tiene una fórmula cerrada: V(estado i) = i / 6. La usas más abajo para medir el error de tus estimaciones.


In [ ]:
# 🎙️ Guion: pide a la clase que calcule a mano V(C), el estado central,
# con la fórmula i / 6 antes de ejecutar la celda (la respuesta es 0.5).
N_ESTADOS_INTERNOS = 5
ESTADO_INICIAL = 3          # C, el estado central de la cadena
TERMINAL_IZQUIERDA = 0
TERMINAL_DERECHA = N_ESTADOS_INTERNOS + 1

# Valor real de cada estado interno: V(estado i) = i / (N_ESTADOS_INTERNOS + 1)
VALORES_REALES = np.array(
    [i / (N_ESTADOS_INTERNOS + 1) for i in range(1, N_ESTADOS_INTERNOS + 1)]
)
print("Valores reales de A, B, C, D, E:", VALORES_REALES)


### Simula un episodio

`generar_episodio` deja que el agente camine desde el estado inicial hasta que cae en un extremo, y devuelve la lista de pares (estado, recompensa) que atravesó. Solo el extremo derecho da recompensa.


In [ ]:
def generar_episodio():
    '''
    Simula un episodio de paseo aleatorio y devuelve la lista de
    (estado, recompensa) que atraviesa, terminando en el estado terminal.
    '''
    estado = ESTADO_INICIAL
    trayectoria = []
    while True:
        # Con probabilidad 0.5 te mueves a la izquierda o a la derecha
        paso = -1 if np.random.rand() < 0.5 else 1
        siguiente_estado = estado + paso
        # Solo el extremo derecho da recompensa; el resto de pasos dan cero
        recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0
        trayectoria.append((estado, recompensa))
        estado = siguiente_estado
        if estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            break
    return trayectoria


### Comprobación rápida

Genera un episodio de ejemplo y observa la secuencia de estados que recorre. Como el paseo es aleatorio, cada ejecución te da un resultado distinto.

> 🎙️ **Guion de clase:** ejecuta la celda dos o tres veces seguidas para que la clase vea que la longitud del episodio cambia cada vez: algunos caminos son cortos, otros dan muchas vueltas antes de terminar.


In [ ]:
np.random.seed(0)
episodio_ejemplo = generar_episodio()
print("Episodio de ejemplo (estado, recompensa):")
print(episodio_ejemplo)


### Monte Carlo de primera visita: la función

`monte_carlo_prediccion` recorre cada episodio de atrás hacia adelante, acumula el retorno G y, para cada estado, se queda con el retorno calculado en su primera aparición. Después actualiza V(s) hacia ese retorno con la regla incremental V(s) ← V(s) + α·(G − V(s)).


In [ ]:
def monte_carlo_prediccion(n_episodios, alpha, gamma=1.0):
    '''
    Estima V para los estados internos 1..5 usando MC de primera visita
    con actualización incremental V(s) <- V(s) + alpha * (G - V(s)).
    '''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)  # inicialización neutra
    historial_rmse = []

    for _ in range(n_episodios):
        trayectoria = generar_episodio()

        # Calcula el retorno G_t desde cada paso hacia el final del episodio
        G = 0.0
        retornos_por_estado = {}
        for estado, recompensa in reversed(trayectoria):
            G = recompensa + gamma * G
            # Primera visita: si el estado se repite, nos quedamos con el
            # retorno calculado en la ocurrencia más temprana (la última
            # que procesamos al recorrer la trayectoria al revés)
            retornos_por_estado[estado] = G

        for estado, G in retornos_por_estado.items():
            if estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                idx = estado - 1  # los estados internos van de 1 a 5
                V[idx] += alpha * (G - V[idx])

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


### Pruébalo: ejecuta Monte Carlo

Ejecuta MC durante 150 episodios con α = 0.1 y observa cuánto se acerca V a los valores reales.


In [ ]:
np.random.seed(0)
V_mc, rmse_mc = monte_carlo_prediccion(n_episodios=150, alpha=0.1)

print("V estimado con MC:", V_mc)
print("Valores reales:   ", VALORES_REALES)
print(f"RMSE final: {rmse_mc[-1]:.4f}")


### TD(0): la función

`td0_prediccion` no espera al final del episodio: en cada paso, calcula el error TD comparando la recompensa observada más tu estimación del siguiente estado con tu estimación actual, y actualiza V(s) al instante.


In [ ]:
def td0_prediccion(n_episodios, alpha, gamma=1.0):
    '''
    Estima V para los estados internos 1..5 con TD(0): actualiza en
    cada paso, sin esperar a que el episodio termine.
    '''
    V = np.full(N_ESTADOS_INTERNOS, 0.5)
    historial_rmse = []

    for _ in range(n_episodios):
        estado = ESTADO_INICIAL
        while estado not in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
            paso = -1 if np.random.rand() < 0.5 else 1
            siguiente_estado = estado + paso
            recompensa = 1.0 if siguiente_estado == TERMINAL_DERECHA else 0.0

            # Valor del siguiente estado: cero si es terminal, si no, la
            # estimación actual (esto es el "bootstrapping")
            if siguiente_estado in (TERMINAL_IZQUIERDA, TERMINAL_DERECHA):
                v_siguiente = 0.0
            else:
                v_siguiente = V[siguiente_estado - 1]

            idx = estado - 1
            error_td = recompensa + gamma * v_siguiente - V[idx]
            V[idx] += alpha * error_td

            estado = siguiente_estado

        rmse = np.sqrt(np.mean((V - VALORES_REALES) ** 2))
        historial_rmse.append(rmse)

    return V, historial_rmse


### Pruébalo: ejecuta TD(0)

Ejecuta TD(0) también durante 150 episodios con α = 0.1, y compara su V final con el de MC.


In [ ]:
np.random.seed(0)
V_td, rmse_td = td0_prediccion(n_episodios=150, alpha=0.1)

print("V estimado con TD(0):", V_td)
print("Valores reales:      ", VALORES_REALES)
print(f"RMSE final: {rmse_td[-1]:.4f}")


## 6. Comparación: velocidad de convergencia

Ejecuta ambos métodos bajo las mismas condiciones y dibuja el RMSE (la raíz del error cuadrático medio, del inglés root mean squared error) frente al valor real, episodio a episodio.


In [ ]:
np.random.seed(0)
N_EPISODIOS = 150
ALPHA = 0.1

_, rmse_mc = monte_carlo_prediccion(N_EPISODIOS, ALPHA)
_, rmse_td = td0_prediccion(N_EPISODIOS, ALPHA)

plt.figure(figsize=(8, 5))
plt.plot(rmse_mc, label="Monte Carlo (primera visita)")
plt.plot(rmse_td, label="TD(0)")
plt.xlabel("Episodios")
plt.ylabel("RMSE frente al valor real")
plt.title("Convergencia de MC vs TD(0) en el paseo aleatorio (α = 0.1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Cómo leer el gráfico

Verás típicamente que TD(0) reduce el error más rápido que MC durante los primeros episodios, gracias a su varianza más baja, aunque ambos acaban acercándose a una zona de error similar a medida que aumentan los episodios. Es la teoría de sesgo y varianza que viste antes, hecha gráfica.

> 🎙️ **Guion de clase:** antes de ejecutar la celda anterior, pregunta "¿qué curva esperáis que baje más rápido al principio, y por qué?". Confirma la intuición con el resultado.


## 7. Experimento en vivo: cambia la tasa de aprendizaje

Prueba tres valores de α (0.01, 0.05 y 0.15) en ambos métodos y observa el efecto. Un α demasiado alto acelera el aprendizaje inicial pero deja más ruido residual; un α demasiado bajo converge de forma más suave, pero mucho más lenta.

> 🎙️ **Guion de clase:** reparte los tres valores de α entre distintas zonas del aula y pide que cada grupo describa en una frase lo que ve en "su" gráfico, antes de comentar los tres juntos.


In [ ]:
np.random.seed(0)
N_EPISODIOS_EXPERIMENTO = 150
ALPHAS_A_PROBAR = [0.01, 0.05, 0.15]

figura, ejes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for eje, alpha in zip(ejes, ALPHAS_A_PROBAR):
    _, rmse_mc_alpha = monte_carlo_prediccion(N_EPISODIOS_EXPERIMENTO, alpha)
    _, rmse_td_alpha = td0_prediccion(N_EPISODIOS_EXPERIMENTO, alpha)
    eje.plot(rmse_mc_alpha, label="MC")
    eje.plot(rmse_td_alpha, label="TD(0)")
    eje.set_title(f"α = {alpha}")
    eje.set_xlabel("Episodios")
    eje.grid(alpha=0.3)

ejes[0].set_ylabel("RMSE frente al valor real")
ejes[0].legend()
plt.tight_layout()
plt.show()


### Más episodios, ¿el mismo error mínimo?

Con α fijo, ninguno de los dos métodos llega exactamente a error cero: la actualización con una tasa de aprendizaje constante deja siempre algo de ruido residual, más alto cuanto mayor es α. Para comprobarlo con claridad, promedia varias ejecuciones con 1.000 episodios y compara el error final de MC y TD(0) para tres valores de α.

> 🎙️ **Guion de clase:** antes de ejecutar, pregunta "si dejo correr esto mil episodios en vez de ciento cincuenta, ¿el error va a llegar a cero?". Es una trampa habitual: con α constante, no lo hace.


In [ ]:
def rmse_promedio(funcion_prediccion, n_episodios, alpha, n_ejecuciones=20):
    '''
    Repite el experimento n_ejecuciones veces con semillas distintas y
    promedia el historial de RMSE, para suavizar el ruido de una sola
    ejecución antes de comparar tasas de aprendizaje.
    '''
    acumulado = np.zeros(n_episodios)
    for ejecucion in range(n_ejecuciones):
        np.random.seed(ejecucion)
        _, historial = funcion_prediccion(n_episodios, alpha)
        acumulado += np.array(historial)
    return acumulado / n_ejecuciones


N_EPISODIOS_LARGO = 1000

print(f"{'α':>6} | {'RMSE final MC':>14} | {'RMSE final TD(0)':>17}")
for alpha in (0.01, 0.05, 0.1):
    rmse_mc_prom = rmse_promedio(monte_carlo_prediccion, N_EPISODIOS_LARGO, alpha)
    rmse_td_prom = rmse_promedio(td0_prediccion, N_EPISODIOS_LARGO, alpha)
    print(f"{alpha:>6} | {rmse_mc_prom[-1]:>14.4f} | {rmse_td_prom[-1]:>17.4f}")


## Puntos clave

- Predicción sin modelo significa estimar Vπ(s) solo a partir de experiencia, sin conocer las probabilidades de transición ni la función de recompensa
- MC promedia retornos completos de episodios; no hace bootstrapping y necesita que el episodio termine
- Primera visita usa solo la primera ocurrencia de un estado por episodio; cada visita usa todas
- TD(0) actualiza tras cada paso con la recompensa observada más tu propia estimación del siguiente estado: eso es bootstrapping
- MC es no sesgado pero tiene varianza alta; TD es sesgado al principio pero tiene varianza baja, y ambos convergen con suficiente experiencia
- Con una tasa de aprendizaje α constante, ningún método llega a error cero: queda un ruido residual que crece con α

## Cierre y próximos pasos

Ya sabes estimar el valor de una política sin conocer el modelo. En la Clase 6 das el siguiente paso: usar esta misma idea de Diferencias Temporales para mejorar la política, no solo evaluarla, con dos algoritmos de control clásicos —SARSA y Q-learning— sobre el entorno de Cliff Walking.

**Antes de la próxima clase:**
- **Abre** el notebook de práctica `clase05_practica_prediccion_sin_modelo.ipynb` y completa los dos ejercicios
- **Experimenta** con otros valores de α y de número de episodios, y anota en qué punto exacto se cruzan las curvas de MC y TD(0)
- **Repasa** el capítulo 6 del manual si quieres profundizar en la derivación matemática

> 🎙️ **Guion de clase:** cierra recordando la frase clave de hoy: "TD ajusta con lo que acaba de pasar más lo que ya creía; MC espera a que termine el viaje entero". Es la distinción que la próxima clase reutiliza para pasar de evaluar políticas a mejorarlas.
